In [ ]:
import os
import pickle
import re
import time
import pandas as pd
from astroquery.utils.tap.core import TapPlus
from astroquery.ipac.ned import Ned
from concurrent.futures import ThreadPoolExecutor, as_completed

# Funciones y configuración inicial

def sanitize(name):
    """Reemplaza caracteres inválidos para Windows por '_'."""
    return re.sub(r'[^A-Za-z0-9_-]', '_', name)

output_dir    = "spectrums NASA"
redshift_path = 'extra/redshift_init_ned.pickle'
os.makedirs(output_dir, exist_ok=True)

# Modificar redshift mínimo
redshift_init = 0
with open(redshift_path, 'wb') as handle:
    pickle.dump(redshift_init, handle)

batch_size_in = 10000      # lote inicial ADQL
max_retries   = 5          # reintentos por lote

all_meta = []             # acumulador de metadatos

# Paginación por redshift para metadatos
while True:

    with open(redshift_path, 'rb') as f:
        redshift_init = float(str(pickle.load(f))[:8])
        print(f"Redshift actual: {redshift_init}")
        print(f"Redshift actual: {redshift_init:.7f}")

    current_batch = batch_size_in
    retries       = 0
    success       = False

    # Intentos de consulta con reducción de lote
    while not success and retries < max_retries:
        adql = f"""
        SELECT TOP {current_batch}
            prefname, z
        FROM NEDTAP.objdir
        WHERE
            z >= {redshift_init:.7f}
            AND z < 9.0
            AND n_spectra > 0
        ORDER BY z
        """
        try:
            tap     = TapPlus(url="https://ned.ipac.caltech.edu/tap")
            batch   = tap.launch_job(adql).get_results().to_pandas()
            success = True

            if batch.empty:
                break

            all_meta.append(batch)

            # Actualizar z_init  
            last_z = float(str(batch['z'].iloc[-1])[:8])
            if last_z == redshift_init:
                last_z += 0.000002
            with open(redshift_path, 'wb') as f:
                pickle.dump(last_z, f)

        except Exception as e:
            retries += 1
            print(e)
            time.sleep(2)

    if not success or batch.empty:
        break

meta_df = pd.concat(all_meta, ignore_index=True)

# Descarga en paralelo con manejo de existentes
def download_spectra(prefname, z):
    base = sanitize(prefname)
    saved = []
    try:
        spectra = Ned.get_spectra(prefname, show_progress=False)
    except Exception:
        return saved

    for idx, hdulist in enumerate(spectra, start=1):
        filename = f"{base}_{idx}_z{z:.5f}.fits"
        path     = os.path.join(output_dir, filename)
        if os.path.exists(path):
            print(f"Saltando (ya existe): {filename}")
        else:
            hdulist.writeto(path, overwrite=True)
            print(f"Guardado: {filename}")
        saved.append(path)
    return saved

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {
        executor.submit(download_spectra, row['prefname'], row['z']): ix
        for ix, row in meta_df.iterrows()
    }
    for future in as_completed(futures):
        pass

print(f"Redshift final: {redshift_init}")

In [ ]:
from astropy.io import fits           # manejo de FITS  
import numpy as np                    # cálculos numéricos  
import matplotlib.pyplot as plt       # visualización  

# Abrir FITS y elegir la extensión que contiene el espectro real
with fits.open('spectrums NASA/WISEA_J020732_20-341640_7______1_z3.00000.fits') as hdul:
    # comprobamos cuál fila varía con espectro
    data = hdul[1].data              # shape = (3, 1024)
    mask_val = np.min(data)          # –2.5e30
    # la fila 1 (índice 1) es la única con señal real
    flux_row = 1
    ivar_row = 2
    header = hdul[1].header

# Reconstruir eje de longitud de onda (en Å)
n_pix = header['NAXIS1']
crval = header['CRVAL1']
crpix = header['CRPIX1']
cdelt = header['CDELT1']
pix   = np.arange(1, n_pix+1)
wave  = (pix - crpix) * cdelt + crval

# Extraer flujo e inversa varianza, enmascarar centinelas
flux = data[flux_row, :].astype(float)
flux[flux == mask_val] = np.nan
ivar = data[ivar_row, :]
error = np.where(ivar>0, 1/np.sqrt(ivar), np.nan)

# Graficar espectro con banda de incertidumbre
plt.figure(figsize=(12,8))
plt.plot(wave, flux, drawstyle='steps-mid')         
plt.fill_between(wave, flux-error, flux+error, alpha=0.3)
plt.xlabel('Longitud de onda (Å)')
plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
obj = header.get('OBJECT', 'WISEA_J144717.48-060020.7')
z   = header.get('Z')
plt.title(f"Espectro de {obj}" + (f", z={z}" if z else ""))
plt.tight_layout()
plt.show()

In [ ]:
from astropy.io import fits           # manejo de FITS  
import numpy as np                    # cálculos numéricos  
import matplotlib.pyplot as plt       # visualización  

# Abrir FITS y extraer la extensión SPECTRUM
with fits.open('spectrums NASA/2MASS_J22384504-2601028________1_z0.00000.fits') as hdul:
    spec_hdu = hdul[1]
    header = spec_hdu.header
    data = spec_hdu.data           # shape = (3, 1024)

# Reconstruir eje de longitud de onda
n_pix = header['NAXIS1']           # 1024 píxeles  
crval = header['CRVAL1']
crpix = header['CRPIX1']
cdelt = header['CDELT1']
pix = np.arange(1, n_pix+1)
wave = (pix - crpix)*cdelt + crval

# Extraer flujo y convertir centinelas a NaN
flux = data[1, :].astype(float)        # fila 1 es el flujo (1024,)
mask_val = data.min()                  # centinela, p.ej. -2.5e30
flux[flux == mask_val] = np.nan        # convertir centinelas en NaN :contentReference[oaicite:1]{index=1}

# Crear máscara booleana: True donde flux es finito (no NaN ni ±inf) :contentReference[oaicite:2]{index=2}
mask = np.isfinite(flux)               # equivalente a ~np.isnan(flux) :contentReference[oaicite:3]{index=3}

# Aplicar la máscara a ambos arrays para eliminar posiciones inválidas :contentReference[oaicite:4]{index=4}
wave_clean = wave[mask]                # sólo longitudes de onda con flujo válido
flux_clean = flux[mask]                # sólo valores de flujo finitos

# (Opcional) reconstruir error asociado si es preciso
ivar = data[2, :]
error = np.where(ivar>0, 1/np.sqrt(ivar), np.nan)
error_clean = error[mask]              # mismo mask para alinear incertidumbres

# Graficar espectro limpio
plt.figure(figsize=(12,8))
plt.plot(wave_clean, flux_clean, drawstyle='steps-mid', label='Flux limpio')
plt.xlabel('Longitud de onda (Å)')
plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
obj = header.get('OBJECT', 'WISEA_J144717.48-060020.7')
z   = header.get('Z')
plt.title(f"Espectro de {obj}" + (f", z={z}" if z else ""))
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Guardado: WISEA_J020732_20-341640_7______1_z3.00000.fits
# Guardado: WISEA_J155112_47_511856_7______1_z3.00012.fits
# Guardado: WISEA_J105242_74_280501_6______1_z3.00013.fits
# Guardado: WISEA_J230037_72-264435_8______1_z3.00024.fits
# Guardado: WISEA_J140218_26-014622_0______1_z3.00043.fits
# Guardado: WISEA_J095107_17_352255_5______1_z3.00043.fits
# Guardado: WISEA_J115419_78_084336_9______1_z3.00048.fits
# Guardado: WISEA_J101329_41_443829_8______1_z3.00053.fits
# Guardado: SDSS_J115240_07_254659_6_______1_z3.00072.fits
# Guardado: SDSS_J021056_34-094631_0_______1_z3.00078.fits
# Guardado: WISEA_J085020_29_131910_6______1_z3.00111.fits
# Guardado: WISEA_J102837_87_203223_6______1_z3.00111.fits
# Guardado: SDSS_J010336_56_002331_6_______1_z3.00125.fits
# Guardado: WISEA_J113429_28_405327_6______1_z3.00129.fits
# Guardado: SDSS_J004232_64_254727_3_______1_z3.00135.fits
# Guardado: WISEA_J101003_96_332625_6______1_z3.00146.fits
# Guardado: WISEA_J151104_77_235335_3______1_z3.00149.fits
# Guardado: _HB89__1615_172________________1_z3.00159.fits
# Guardado: WISEA_J082543_24_383829_3______1_z3.00163.fits
# Guardado: WISEA_J101943_19_061935_2______1_z3.00167.fits
# Guardado: SDSS_J091019_78_240330_0_______1_z3.00175.fits
# Guardado: WISEA_J120857_73_073706_0______1_z3.00187.fits
# Guardado: WISEA_J143825_57_315126_2______1_z3.00215.fits
# Guardado: WISEA_J124359_62_633606_7______1_z3.00224.fits
# Guardado: WISEA_J125356_89_111355_2______1_z3.00234.fits
# Guardado: WISEA_J130329_45_060307_4______1_z3.00244.fits
# Guardado: WISEA_J092434_42_435512_8______1_z3.00259.fits
# Guardado: SDSS_J151155_89-015106_9_______1_z3.00265.fits
# Guardado: WISEA_J110427_07_054848_3______1_z3.00272.fits
# Guardado: SDSS_J022345_41-003235_6_______1_z3.00278.fits
# Guardado: WISEA_J131550_41_463537_3______1_z3.00281.fits
# Guardado: WISEA_J085833_02_401203_1______1_z3.00283.fits
# Guardado: WISEA_J085833_02_401203_1______2_z3.00283.fits
# Guardado: WISEA_J124002_66_443205_8______1_z3.00289.fits
# Guardado: WISEA_J142133_37_392241_7______1_z3.00297.fits
# Guardado: WISEA_J092322_89_033820_9______1_z3.00299.fits
# Guardado: WISEA_J091009_07_535355_6______1_z3.00316.fits
# Guardado: WISEA_J073706_77_430903_4______1_z3.00328.fits
# Guardado: WISEA_J125405_19_352813_6______1_z3.00328.fits
# Guardado: WISEA_J144135_29_105221_9______1_z3.00344.fits
# Guardado: WISEA_J131624_00-015835_0______1_z3.00348.fits
# Guardado: WISEA_J115213_13_313839_5______1_z3.00357.fits
# Guardado: SDSS_J111610_68_411814_4_______1_z3.00361.fits
# Guardado: WISEA_J103126_13_405532_6______1_z3.00387.fits
# Guardado: SDSS_J103032_12_134919_1_______1_z3.00412.fits
# Guardado: WISEA_J014216_33-295507_4______1_z3.00420.fits
# Guardado: WISEA_J073735_51_280927_5______1_z3.00421.fits
# Guardado: WISEA_J205516_83-051110_9______1_z3.00422.fits









from astropy.io import fits           # manejo de FITS  
import numpy as np                    # cálculos numéricos  
import matplotlib.pyplot as plt       # visualización  

# Abrir FITS y elegir la extensión que contiene el espectro real
hdul = fits.open('spectrums NASA/WISEA_J205516_83-051110_9______1_z3.00422.fits')
info = hdul.info()
print(info)
header = hdul[0].header
print(header)
header = hdul[1].header
print(header)
header = hdul[2].header
print(header)
header = hdul[3].header
print(header)
header = hdul[4].header
print(header)
header = hdul[5].header
print(header)

data    = hdul[0].data      # shape (5,3856)
header0 = hdul[0].header

# Construir vector de longitudes de onda (log‑linear)
n_pix  = header0['NAXIS1']
coeff0 = header0['COEFF0']
coeff1 = header0['COEFF1']
crpix1 = header0.get('CRPIX1', 1)
pix    = np.arange(n_pix)
wave   = 10**(coeff0 + coeff1 * (pix + 1 - crpix1))

# Extraer flujo y error (fila-wise)
flux  = data[1, :]
error = data[2, :]

# Graficar
plt.figure(figsize=(12, 8))
plt.plot(wave, flux, drawstyle='steps-mid', label='Flujo')
plt.fill_between(wave, flux-error, flux+error, alpha=0.3, label='±1σ')
plt.xlabel('Longitud de onda (Å)')
plt.ylabel('Flujo (10⁻¹⁷ erg s⁻¹ cm⁻² Å⁻¹)')
obj = header0.get('OBJECT', 'WISEA_J144717.48-060020.7')
z   = header0.get('Z')
plt.title(f"Espectro de {obj}" + (f", z={z:.3f}" if z else ""))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============
# CARGA DE LIBRERIAS

from astropy.io import fits
import gc
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
import torch
import torch.nn as nn
import torch.optim as optim
import random
import pickle
import numpy as np
import torch
from astropy.io import fits
import matplotlib.pyplot as plt
from astroquery.ipac.ned import Ned


# =============
# DEFINICIÓN Y CONFIGURACIÓN DEL MODELO

# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Transformer(nn.Module):
    def __init__(self, num_points, d_model=128, nhead=8, num_layers=4, dim_feedforward=256):
        super(Transformer, self).__init__()
        self.embedding = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2)
        )
        reduced_len = num_points // 16
        self.positional_encoding = PositionalEncoding(d_model, dropout=0.1, max_len=reduced_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=0.1, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc_layers = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.transpose(1, 2)
        x = self.pooling(x).squeeze(-1)
        out = self.fc_layers(x)
        return out

# Instanciar el modelo y moverlo a GPU si está disponible
num_points = 5000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelTransformer = Transformer(num_points).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(modelTransformer.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

print("Modelo Transformer configurado correctamente.")




flux = flux
test_redshift=z




# Expandir/pad a num_points y escalado  
def expand_points(wl, fl, target_count):
    wl, fl = list(wl), list(fl)
    diffs = [abs(fl[i+1]-fl[i]) for i in range(len(fl)-1)]
    for idx in sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True):
        if len(wl)>=target_count: break
        wl.insert(idx+1, (wl[idx]+wl[idx+1])/2)
        fl.insert(idx+1, (fl[idx]+fl[idx+1])/2)
    return np.array(wl), np.array(fl)

num_points = 5000
wavelength, flux = expand_points(wave, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)

with open('extra/scaler_fitted.pkl','rb') as f:
    scalers = pickle.load(f)
flux_scaler = scalers["flux_scaler"]
wave_scaler = scalers["wavelength_scaler"]

# Normalización y tensor de entrada
f_in = flux.reshape(1,-1)
w_in = wavelength.reshape(1,-1)
f_s = flux_scaler.transform(f_in)
w_s = wave_scaler.transform(w_in)
inp = np.stack([f_s[0], w_s[0]], axis=0).reshape(1,2,num_points)
input_tensor = torch.from_numpy(inp).to(device)
input_tensor = input_tensor.float()

# Carga modelo y predicción
checkpoint = torch.load('storage/modelTRA_1M.pth', map_location=device)
modelTransformer.load_state_dict(checkpoint['model_state_dict'])
modelTransformer.eval()
with torch.no_grad():
    pred_z = modelTransformer(input_tensor).item()

print("Redshift real:    ", test_redshift)
print("Redshift predicho:", pred_z)

# Gráficos
plt.figure(figsize=(12, 6))
plt.plot(wavelength, flux, label='Original') 
plt.xlabel('λ (Å)'); plt.ylabel('Flujo'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(w_s.flatten(), f_s.flatten(), label="Espectro Normalizado", color="orange")
plt.xlabel("Longitud de onda Normalizada")
plt.ylabel("Flujo Normalizado")
plt.title("Espectro Normalizado")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# =============
# CARGA DE LIBRERIAS

from astropy.io import fits
import gc
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
import torch
import torch.nn as nn
import torch.optim as optim
import random
import pickle
import numpy as np
import torch
from astropy.io import fits
import matplotlib.pyplot as plt
from astroquery.ipac.ned import Ned


# =============
# DEFINICIÓN Y CONFIGURACIÓN DEL MODELO

# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Transformer(nn.Module):
    def __init__(self, num_points, d_model=128, nhead=8, num_layers=4, dim_feedforward=256):
        super(Transformer, self).__init__()
        self.embedding = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.MaxPool1d(kernel_size=2)
        )
        reduced_len = num_points // 16
        self.positional_encoding = PositionalEncoding(d_model, dropout=0.1, max_len=reduced_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=0.1, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.fc_layers = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = x.transpose(1, 2)
        x = self.pooling(x).squeeze(-1)
        out = self.fc_layers(x)
        return out

# Instanciar el modelo y moverlo a GPU si está disponible
num_points = 5000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelTransformer = Transformer(num_points).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(modelTransformer.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

print("Modelo Transformer configurado correctamente.")




flux = flux
test_redshift=z




# Expandir/pad a num_points y escalado  
def expand_points(wl, fl, target_count):
    wl, fl = list(wl), list(fl)
    diffs = [abs(fl[i+1]-fl[i]) for i in range(len(fl)-1)]
    for idx in sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True):
        if len(wl)>=target_count: break
        wl.insert(idx+1, (wl[idx]+wl[idx+1])/2)
        fl.insert(idx+1, (fl[idx]+fl[idx+1])/2)
    return np.array(wl), np.array(fl)

num_points = 5000
wavelength, flux = expand_points(wave, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)
wavelength, flux = expand_points(wavelength, flux, num_points)

# Normalización y tensor de entrada

if flux.ndim == 1:  
    flux = flux[np.newaxis, :]
    wavelength = wavelength[np.newaxis, :] 


f_s = flux / flux.max(axis=1, keepdims=True)  
wave_shifted = wavelength - wavelength.mean(axis=1, keepdims=True)  
w_s = wave_shifted / np.abs(wave_shifted).max(axis=1, keepdims=True)

inp = np.stack([f_s[0], w_s[0]], axis=0).reshape(1,2,num_points)
input_tensor = torch.from_numpy(inp).to(device)
input_tensor = input_tensor.float()


# Carga modelo y predicción
checkpoint = torch.load('storage/modelTRA_1M_renorm.pth', map_location=device)
modelTransformer.load_state_dict(checkpoint['model_state_dict'])
modelTransformer.eval()
with torch.no_grad():
    pred_z = modelTransformer(input_tensor).item()

print("Redshift real:    ", test_redshift)
print("Redshift predicho:", pred_z)

# Gráficos
plt.figure(figsize=(12, 6))
plt.plot(wavelength[0], flux[0], label='Original') 
plt.xlabel('λ (Å)'); plt.ylabel('Flujo'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(w_s.flatten(), f_s.flatten(), label="Espectro Normalizado", color="orange")
plt.xlabel("Longitud de onda Normalizada")
plt.ylabel("Flujo Normalizado")
plt.title("Espectro Normalizado")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()